In [2]:
import langgraph
from langgraph.graph import StateGraph, END
print("langgraph imported successfully")

langgraph imported successfully


In [3]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

# this is the shared dictionary every node reads and writes
class ToyState(TypedDict):
    input_text: str
    word_count: int
    is_long: bool
    summary: str
    loop_count: int

print("state schema defined")
print("every node in the graph will receive this dict and return an updated version")

state schema defined
every node in the graph will receive this dict and return an updated version


In [4]:
# each node is just a plain python function
# it receives the current state and returns a dict of keys to update

def count_words(state: ToyState) -> dict:
    text = state['input_text']
    count = len(text.split())
    print(f"  [count_words] counted {count} words")
    return {
        "word_count": count,
        "loop_count": state.get('loop_count', 0)
    }


def check_length(state: ToyState) -> dict:
    is_long = state['word_count'] > 20
    print(f"  [check_length] is_long = {is_long}")
    return {"is_long": is_long}


def summarize(state: ToyState) -> dict:
    text = state['input_text']
    # simple truncation as a fake summary
    summary = text[:50] + "..." if len(text) > 50 else text
    print(f"  [summarize] produced summary")
    return {"summary": f"SUMMARY: {summary}"}


def expand(state: ToyState) -> dict:
    # pretend to expand a short text
    count = state.get('loop_count', 0) + 1
    print(f"  [expand] loop count now {count}")
    expanded = state['input_text'] + " [additional context added by agent]"
    return {
        "input_text": expanded,
        "loop_count": count
    }


print("all nodes defined")

all nodes defined


In [5]:
# this is what makes it a LOOP and not just a pipeline
# based on state, it decides which node to go to next

def route_after_check(state: ToyState) -> str:
    if state['loop_count'] >= 2:
        # safety guard — never loop more than twice
        print("  [router] max loops reached, forcing summarize")
        return "summarize"

    if state['is_long']:
        print("  [router] text is long, going to summarize")
        return "summarize"
    else:
        print("  [router] text is short, going to expand then recheck")
        return "expand"

print("router defined")

router defined


In [6]:
graph = StateGraph(ToyState)

# add all nodes
graph.add_node("count_words",  count_words)
graph.add_node("check_length", check_length)
graph.add_node("summarize",    summarize)
graph.add_node("expand",       expand)

# set entry point
graph.set_entry_point("count_words")

# add fixed edges
graph.add_edge("count_words",  "check_length")

# add conditional edge — router decides where to go
graph.add_conditional_edges(
    "check_length",
    route_after_check,
    {
        "summarize": "summarize",
        "expand":    "expand"
    }
)

# after expand, go back to count_words — this is the loop
graph.add_edge("expand", "count_words")

# summarize is the final step
graph.add_edge("summarize", END)

app = graph.compile()
print("graph compiled successfully")

graph compiled successfully


In [7]:
print("=== Test 1: short text (should loop) ===\n")

result = app.invoke({
    "input_text": "The sky is blue.",
    "word_count": 0,
    "is_long": False,
    "summary": "",
    "loop_count": 0
})

print("\nFinal state:")
print("  word_count:", result['word_count'])
print("  is_long:   ", result['is_long'])
print("  loop_count:", result['loop_count'])
print("  summary:   ", result['summary'])

=== Test 1: short text (should loop) ===

  [count_words] counted 4 words
  [check_length] is_long = False
  [router] text is short, going to expand then recheck
  [expand] loop count now 1
  [count_words] counted 9 words
  [check_length] is_long = False
  [router] text is short, going to expand then recheck
  [expand] loop count now 2
  [count_words] counted 14 words
  [check_length] is_long = False
  [router] max loops reached, forcing summarize
  [summarize] produced summary

Final state:
  word_count: 14
  is_long:    False
  loop_count: 2
  summary:    SUMMARY: The sky is blue. [additional context added by agen...


In [8]:
print("=== Test 2: long text (should go straight to summarize) ===\n")

long_text = (
    "Deep learning models are powerful function approximators that learn "
    "hierarchical representations from raw data through stacked layers of "
    "nonlinear transformations trained end to end via backpropagation."
)

result2 = app.invoke({
    "input_text": long_text,
    "word_count": 0,
    "is_long": False,
    "summary": "",
    "loop_count": 0
})

print("\nFinal state:")
print("  word_count:", result2['word_count'])
print("  is_long:   ", result2['is_long'])
print("  loop_count:", result2['loop_count'])
print("  summary:   ", result2['summary'])

=== Test 2: long text (should go straight to summarize) ===

  [count_words] counted 26 words
  [check_length] is_long = True
  [router] text is long, going to summarize
  [summarize] produced summary

Final state:
  word_count: 26
  is_long:    True
  loop_count: 0
  summary:    SUMMARY: Deep learning models are powerful function approxi...


In [9]:
print("""
What you just built and why it matters:

1. StateGraph    — a graph where nodes share one dict (the state)
2. Nodes         — plain python functions that read state and return updates
3. Fixed edges   — always go from A to B
4. Conditional   — go to A or B based on what the state contains
5. The loop      — expand goes back to count_words, creating a cycle
6. END           — the graph stops when it reaches this

This is EXACTLY the structure your XAI agent will use:
  - Planner node  decides which tool to run (conditional edge)
  - SHAP node     runs and writes results to state
  - Grad-CAM node runs and writes results to state
  - Critic node   checks for contradictions (conditional edge back to Planner)
  - Narrator node writes the final explanation and goes to END

The only difference is your nodes will call real ML tools and LLMs
instead of counting words.
""")


What you just built and why it matters:

1. StateGraph    — a graph where nodes share one dict (the state)
2. Nodes         — plain python functions that read state and return updates
3. Fixed edges   — always go from A to B
4. Conditional   — go to A or B based on what the state contains
5. The loop      — expand goes back to count_words, creating a cycle
6. END           — the graph stops when it reaches this

This is EXACTLY the structure your XAI agent will use:
  - Planner node  decides which tool to run (conditional edge)
  - SHAP node     runs and writes results to state
  - Grad-CAM node runs and writes results to state
  - Critic node   checks for contradictions (conditional edge back to Planner)
  - Narrator node writes the final explanation and goes to END

The only difference is your nodes will call real ML tools and LLMs
instead of counting words.



In [11]:
# this is the real state you will use from Day 9 onwards
# write it here so it is in version control

from typing import TypedDict, Optional, List

class XAIAgentState(TypedDict):
    # input
    image_path:      str
    image_tensor:    object   # torch.Tensor — not typed strictly here

    # model output
    prediction:      str
    confidence:      float
    pred_class_idx:  int

    # tool outputs
    shap_result:     dict
    gradcam_result:  dict

    # agent reasoning
    critique:        str
    contradictions:  List[str]
    next_action:     str

    # final output
    explanation:     str
    confidence_note: str

    # loop control
    loop_count:      int

print("XAIAgentState defined")
print("this is the shared dictionary your 5 agent nodes will use")


XAIAgentState defined
this is the shared dictionary your 5 agent nodes will use
